In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [ ]:
# 1. Carregar os dados
df_dados = pd.read_excel("Base Leite 2023-2024 Formatada.xlsx")
df_dados

In [ ]:
df_dados.groupby('sistema')['id_fazenda'].nunique()

In [ ]:
df_dados.info()

In [ ]:
df_dados = df_dados.drop_duplicates()

In [ ]:
def create_dataset(sistema, grupo, ano):
    df_ano = df_dados[df_dados["ano"] == ano]

    if sistema == "cf":
        df_ano_sis = df_ano.loc[
            (df_ano["sistema"] == "compost-barn") |
            (df_ano["sistema"] == "free-stall")
        ]
    else:
        df_ano_sis = df_ano.loc[
            (df_ano["sistema"] == sistema)
        ]
    
    if grupo == "inf":
        relacao_troca = df_ano_sis.loc[(df_ano_sis["52_custo_total_do_leite_r_litro"] >= df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.75)), "35_relacao_de_troca_leite_concentrado_kg_l"].mean()
        percentual_gasto_volumoso = df_ano_sis.loc[(df_ano_sis["52_custo_total_do_leite_r_litro"] >= df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.75)), "60_gasto_com_volumoso_na_atividade_renda_bruta_da_atividade_percentual"].mean()
        percentual_gasto_maodeobra = df_ano_sis.loc[(df_ano_sis["52_custo_total_do_leite_r_litro"] >= df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.75)), "59_gasto_com_mao_de_obra_contratada_na_atividade_renda_bruta_da_atividade_percentual"].mean()
        percentual_gasto_concentrado = df_ano_sis.loc[(df_ano_sis["52_custo_total_do_leite_r_litro"] >= df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.75)), "61_gasto_com_concentrado_na_atividade_renda_bruta_da_atividade_percentual"].mean()
        idx_grupo = 0

        df_ano_sis = df_ano_sis.loc[(df_ano_sis["52_custo_total_do_leite_r_litro"] >= df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.75))]
    elif grupo == "mid":
        relacao_troca = df_ano_sis.loc[
            (df_ano_sis["52_custo_total_do_leite_r_litro"] > df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.25)) &
            (df_ano_sis["52_custo_total_do_leite_r_litro"] < df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.75)),
            "35_relacao_de_troca_leite_concentrado_kg_l"
        ].mean()
        percentual_gasto_volumoso = df_ano_sis.loc[
            (df_ano_sis["52_custo_total_do_leite_r_litro"] > df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.25)) &
            (df_ano_sis["52_custo_total_do_leite_r_litro"] < df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.75)),
            "60_gasto_com_volumoso_na_atividade_renda_bruta_da_atividade_percentual"
        ].mean()
        percentual_gasto_maodeobra = df_ano_sis.loc[
            (df_ano_sis["52_custo_total_do_leite_r_litro"] > df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.25)) &
            (df_ano_sis["52_custo_total_do_leite_r_litro"] < df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.75)),
            "59_gasto_com_mao_de_obra_contratada_na_atividade_renda_bruta_da_atividade_percentual"
        ].mean()
        percentual_gasto_concentrado = df_ano_sis.loc[
            (df_ano_sis["52_custo_total_do_leite_r_litro"] > df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.25)) &
            (df_ano_sis["52_custo_total_do_leite_r_litro"] < df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.75)),
            "61_gasto_com_concentrado_na_atividade_renda_bruta_da_atividade_percentual"
        ].mean()

        idx_grupo = 1

        df_ano_sis = df_ano_sis.loc[
            (df_ano_sis["52_custo_total_do_leite_r_litro"] > df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.25)) &
            (df_ano_sis["52_custo_total_do_leite_r_litro"] < df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.75))
        ]
    else:
        relacao_troca = df_ano_sis.loc[(df_ano_sis["52_custo_total_do_leite_r_litro"] <= df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.25)), "35_relacao_de_troca_leite_concentrado_kg_l"].mean()
        percentual_gasto_volumoso = df_ano_sis.loc[(df_ano_sis["52_custo_total_do_leite_r_litro"] <= df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.25)), "60_gasto_com_volumoso_na_atividade_renda_bruta_da_atividade_percentual"].mean()
        percentual_gasto_maodeobra = df_ano_sis.loc[(df_ano_sis["52_custo_total_do_leite_r_litro"] <= df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.25)), "59_gasto_com_mao_de_obra_contratada_na_atividade_renda_bruta_da_atividade_percentual"].mean()
        percentual_gasto_concentrado = df_ano_sis.loc[(df_ano_sis["52_custo_total_do_leite_r_litro"] >= df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.25)), "61_gasto_com_concentrado_na_atividade_renda_bruta_da_atividade_percentual"].mean()

        idx_grupo = 2

        df_ano_sis = df_ano_sis.loc[(df_ano_sis["52_custo_total_do_leite_r_litro"] <= df_ano_sis["52_custo_total_do_leite_r_litro"].quantile(0.25))]

    df_result = pd.DataFrame()
    
    # Entradas do usuario
    df_result["vacas_lactacao"] = df_ano_sis["6_vacas_em_lactacao_animais_mes"]
    df_result["total_vacas"] = df_ano_sis["7_total_de_vacas_animais_mes"]
    df_result["producao_vacas_lactacao_L_dia"] = df_ano_sis["25_producao_vacas_em_lactacao_litros_dia"]
    df_result["preco_concentrado_r_kg"] = df_ano_sis["34_preco_medio_do_concentrado_r_kg"]
    df_result["preco_leite_r_L"] = df_ano_sis["33_preco_medio_do_leite_r_litro"]
    df_result["qtde_maodeobra"] = df_ano_sis["10_mao_de_obra_total_trabalhador_dia"]
    df_result["area_destinada_atividade"] = df_ano_sis["1_area_destinada_a_atividade_hectare"]

    # Medias do grupo
    df_result.loc[:, "relacao_troca_kg_L"] = relacao_troca
    df_result.loc[:, "percentual_gasto_volumoso"] = percentual_gasto_volumoso
    df_result.loc[:, "percentual_gasto_maodeobra"] = percentual_gasto_maodeobra
    df_result.loc[:, "percentual_gasto_concentrado"] = percentual_gasto_concentrado
    df_result.loc[:, "grupo"] = idx_grupo

    # Variaveis calculadas
    df_result["percentual_vacas_lactacao"] = (df_result["vacas_lactacao"] / df_result["total_vacas"]) * 100
    df_result["producao_leite_anual"] = (df_result["producao_vacas_lactacao_L_dia"] * 30) * df_result["vacas_lactacao"] * 12
    # df_result["consumo_concentrado_kg_ano"] = df_result["producao_leite_anual"] * df_result["relacao_troca_kg_L"]
    # df_result["gasto_concentrado"] = df_result["consumo_concentrado_kg_ano"] * df_result["preco_concentrado_r_kg"]
    # df_result["renda_bruta"] = df_result["producao_leite_anual"] * df_result["preco_leite_r_L"]
    # df_result["percentual_gasto_concentrado"] = (df_result["gasto_concentrado"] / df_result["renda_bruta"]) * 100
    df_result["producao_total_vacas"] = (df_result["producao_leite_anual"] / 365) / df_result["total_vacas"]
    df_result["producao_maodeobra"] = (df_result["producao_leite_anual"] / 365) / df_result["qtde_maodeobra"]
    df_result["vacas_lactacao_maodeobra"] = df_result["vacas_lactacao"] / df_result["qtde_maodeobra"]
    df_result["vacas_lactacao_area"] = df_result["vacas_lactacao"] / df_result["area_destinada_atividade"]

    # features
    df_result["id_fazenda"] = df_ano_sis["id_fazenda"]
    df_result["sistema"] = df_ano_sis["sistema"]
    df_result["ano"] = df_ano_sis["ano"]
    df_result["18_vacas_em_lactacao_total_de_animais_percentual"] = df_result["percentual_vacas_lactacao"]
    df_result["20_vacas_em_lactacao_area_destinada_a_atividade_considerando_reserva_animais_hectare"] = df_result["vacas_lactacao_area"]
    df_result["21_vacas_em_lactacao_mao_de_obra_total_animais_trabalhador_dia"] = df_result["vacas_lactacao_maodeobra"]
    df_result["26_producao_total_de_vacas_litros_dia"] = df_result["producao_total_vacas"]
    df_result["27_producao_mao_de_obra_total_litros_trabalhador_dia"] = df_result["producao_maodeobra"]
    df_result["33_preco_medio_do_leite_r_litro"] = df_result["preco_leite_r_L"]
    df_result["35_relacao_de_troca_leite_concentrado_kg_l"] = df_result["relacao_troca_kg_L"]
    df_result["59_gasto_com_mao_de_obra_contratada_na_atividade_renda_bruta_da_atividade_percentual"] = df_result["percentual_gasto_maodeobra"]
    df_result["60_gasto_com_volumoso_na_atividade_renda_bruta_da_atividade_percentual"] = df_result["percentual_gasto_volumoso"]
    df_result["61_gasto_com_concentrado_na_atividade_renda_bruta_da_atividade_percentual"] = df_result["percentual_gasto_concentrado"]

    # target
    df_result["52_custo_total_do_leite_r_litro"] = df_ano_sis["52_custo_total_do_leite_r_litro"]

    return df_result

In [ ]:
df_all = pd.DataFrame()

In [ ]:
anos = [2023,2024]
sistemas = ["cf", "confinado-sem-estrutura", "semiconfinado"]
grupos = ["sup"]

for ano in anos:
    for sistema in sistemas:
        for grupo in grupos:
            df_output = create_dataset(sistema, grupo, ano)
        df_all = pd.concat([df_all, df_output])

In [ ]:
"""
consumo_concentrado = producao_leite_anual (produtividade_vaca_em_lactacao_dia (entrada) * percentual_vacas_lactacao (calculado abaixo) * total_vacas (entrada) * 365) * relacao_troca (kg/L) (media do grupo)
gasto_concentrado = consumo_concentrado * preco_concentrado (entrada)
percentual_gasto_concentrado = (gasto_concentrado / renda_bruta (preco_leite (entrada) * producao_leite_anual)) * 100

gasto_volumoso_percentual = media_do_grupo (Inferiores = 25% dos produtores com maior custo por L de leite; Intermediarios = 50%; Superiores = 25% dos produtores com menor...)
gasto_maodeobra_percentual = media_do_grupo (Inferiores = 25% dos produtores com maior custo por L de leite; Intermediarios = 50%; Superiores = 25% dos produtores com menor...)

relacao_troca (kg/L) = media_do_grupo

qtde_maodeobra = entrada
producao_maodeobra = producao_leite_diaria (calculada) / qtde_maodeobra
percentual_vacas_lactacao = vacas_lactacao (entrada) / total_vacas (entrada) * 100

area_destinada_atividade = entrada
"""

In [ ]:
# 2. Definir variáveis de entrada (features) e a variável alvo (target)
features_standard = [
    "61_gasto_com_concentrado_na_atividade_renda_bruta_da_atividade_percentual",
    "60_gasto_com_volumoso_na_atividade_renda_bruta_da_atividade_percentual",
    "27_producao_mao_de_obra_total_litros_trabalhador_dia",
    "18_vacas_em_lactacao_total_de_animais_percentual",
    "26_producao_total_de_vacas_litros_dia"
]

features_minmax = [
    "59_gasto_com_mao_de_obra_contratada_na_atividade_renda_bruta_da_atividade_percentual",
    "21_vacas_em_lactacao_mao_de_obra_total_animais_trabalhador_dia"
]

features_robust = [
    "35_relacao_de_troca_leite_concentrado_kg_l",
    "33_preco_medio_do_leite_r_litro",
    "20_vacas_em_lactacao_area_destinada_a_atividade_considerando_reserva_animais_hectare"
    
]

features_cat = ["id_fazenda", "sistema", "grupo"]

target = "52_custo_total_do_leite_r_litro"

todas_features = features_standard + features_minmax + features_robust + features_cat
data = df_all[todas_features + ['ano', target]].dropna()

train_data = data[data['ano'] == 2023]
test_data = data[data['ano'] == 2024]

# O 'ano' NÃO entra no X_train nem no X_test, pois ele é constante dentro de cada grupo
X_train = train_data[todas_features]
y_train = train_data[target]

X_test = test_data[todas_features]
y_test = test_data[target]

# 4. Criar o pré-processador
# Variáveis numéricas: são padronizadas (StandardScaler)
# Variáveis categóricas: são transformadas em colunas binárias (OneHotEncoder)
preprocessor = ColumnTransformer(
    transformers=[
        ('std_scaler', StandardScaler(), features_standard),
        ('minmax_scaler', MinMaxScaler(), features_minmax),
        ('robust_scaler', RobustScaler(), features_robust),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_cat)
    ]
)

# 5. Dicionário com os Modelos de Machine Learning que serão testados
models = {
    "Regressão Linear": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=150, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=150, random_state=42),
    "Rede Neural (Rasa - 64 neurônios)": MLPRegressor(
        hidden_layer_sizes=(64,), solver='lbfgs', max_iter=2000, random_state=42
    ),
    "Rede Neural (Média - 64x32)": MLPRegressor(
        hidden_layer_sizes=(64, 32), solver='lbfgs', max_iter=2000, random_state=42
    ),
    "Rede Neural (Profunda - 128x64x32)": MLPRegressor(
        hidden_layer_sizes=(128, 64, 32), solver='lbfgs', max_iter=2000, random_state=42
    ),
    "Regressão Ridge (Penalizada)": Ridge(alpha=1.0),
    "Elastic Net": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "Support Vector Regression (SVR)": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    "LightGBM (HistGradientBoosting)": HistGradientBoostingRegressor(random_state=42)
}

# Lista para salvar os resultados
results = []

# 7. Treinamento e Avaliação
for name, model in models.items():
    # Cria o pipeline que vai rodar o pré-processamento antes de entregar ao modelo
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Treina o modelo
    pipeline.fit(X_train, y_train)
    
    # Previsões nos dados de teste
    y_pred = pipeline.predict(X_test)
    
    # Cálculos de Erros
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Modelo": name,
        "MAE (R$/L)": round(mae, 4),
        "RMSE (R$/L)": round(rmse, 4),
        "R²": round(r2, 4)
    })

# 8. Mostrar a tabela com os resultados finais
results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))

In [ ]:
data[[
    "id_fazenda",
    "sistema",
    "grupo",
    "ano",
    "52_custo_total_do_leite_r_litro",
    "18_vacas_em_lactacao_total_de_animais_percentual",
    "20_vacas_em_lactacao_area_destinada_a_atividade_considerando_reserva_animais_hectare",
    "21_vacas_em_lactacao_mao_de_obra_total_animais_trabalhador_dia",
    "26_producao_total_de_vacas_litros_dia",
    "27_producao_mao_de_obra_total_litros_trabalhador_dia",
    "33_preco_medio_do_leite_r_litro",
    "35_relacao_de_troca_leite_concentrado_kg_l",
    "59_gasto_com_mao_de_obra_contratada_na_atividade_renda_bruta_da_atividade_percentual",
    "60_gasto_com_volumoso_na_atividade_renda_bruta_da_atividade_percentual", 
    "61_gasto_com_concentrado_na_atividade_renda_bruta_da_atividade_percentual"
]]

In [ ]:
data[[
    "id_fazenda",
    "sistema",
    "grupo",
    "ano",
    "52_custo_total_do_leite_r_litro",
    "18_vacas_em_lactacao_total_de_animais_percentual",
    "20_vacas_em_lactacao_area_destinada_a_atividade_considerando_reserva_animais_hectare",
    "21_vacas_em_lactacao_mao_de_obra_total_animais_trabalhador_dia",
    "26_producao_total_de_vacas_litros_dia",
    "27_producao_mao_de_obra_total_litros_trabalhador_dia",
    "33_preco_medio_do_leite_r_litro",
    "35_relacao_de_troca_leite_concentrado_kg_l",
    "59_gasto_com_mao_de_obra_contratada_na_atividade_renda_bruta_da_atividade_percentual",
    "60_gasto_com_volumoso_na_atividade_renda_bruta_da_atividade_percentual", 
    "61_gasto_com_concentrado_na_atividade_renda_bruta_da_atividade_percentual"
]][data["ano"] == 2023].describe()